# Latihan Formatif &middot; Pertemuan 12

## Menghitung Forward Pass dan Menelusuri Alur Backpropagation

**Mata Kuliah** Data Science (TI24425) &middot; 3 sks (Teori)
**Program Studi** Teknologi Informasi &middot; Politeknik Negeri Madiun
**Rujukan** Sub-CPMK 11 &middot; Arsitektur Neural Network
**Perkiraan waktu** 60–90 menit

Latihan ini menagih kembali perhitungan forward pass pada jaringan kecil, lalu menelusuri arah mundur backpropagation secara konseptual.

---

| | |
|---|---|
| **Nama** | |
| **NPM** | |
| **Kelas** | |
| **Tanggal dikerjakan** | |


---

## Petunjuk

### Sifat latihan ini

Latihan ini **bersifat formatif dan tidak berbobot tersendiri**. Tidak ada nilai yang diberikan untuk berkas ini.

Yang dinilai adalah **kontribusi Anda saat pembahasan di kelas**, melalui komponen **Aktivitas Partisipatif**. Karena itu, yang penting bukan benar atau salahnya jawaban Anda, melainkan apakah Anda datang ke kelas sudah mencoba dan sudah tahu di bagian mana Anda tersendat.

### Cara mengerjakan

1. **Kerjakan dengan tangan lebih dulu.** Sediakan kertas dan kalkulator sederhana. Jangan langsung menjalankan sel kode.
2. Isi jawaban Anda pada variabel yang disediakan di sel `[PERIKSA]`.
3. Jalankan sel tersebut. Anda akan diberi tahu **cocok** atau **belum cocok**, tanpa diberi jawabannya.
4. Bila belum cocok, periksa ulang langkah Anda. Jangan mengubah angka sampai kebetulan cocok — itu tidak melatih apa pun.
5. Bagian **Pertanyaan untuk Dibawa ke Kelas** tidak punya pemeriksa. Bagian itulah yang akan dibahas bersama.

### Bila tetap tersendat

Tuliskan di **Lembar Catatan** pada akhir berkas: langkah keberapa Anda tersendat dan apa yang membingungkan. Catatan itu jauh lebih berguna dibawa ke kelas daripada jawaban kosong.


---

## Persiapan

In [ ]:
# [KODE] Alat pemeriksa jawaban — jalankan sekali di awal
import hashlib
import numpy as np
import matplotlib.pyplot as plt

def periksa(label, jawaban_anda, sandi_benar, desimal=3):
    if jawaban_anda is None:
        print(f'  ○ {label:38s} belum diisi')
        return
    cap = hashlib.sha256(f'{round(float(jawaban_anda), desimal):.{desimal}f}'.encode()).hexdigest()[:16]
    if cap == sandi_benar:
        print(f'  ✓ {label:38s} cocok')
    else:
        print(f'  ✗ {label:38s} belum cocok — periksa kembali langkah Anda')

print('Alat pemeriksa siap. Isi jawaban Anda pada sel [PERIKSA], lalu jalankan.')

---

# Soal 1 &mdash; Forward Pass pada Jaringan 2–2–1

**Arsitektur.** Dua masukan, dua neuron tersembunyi dengan aktivasi **ReLU**, satu neuron keluaran dengan aktivasi **sigmoid**.

**Bobot dan bias:**

| Neuron | Bobot | Bias |
|---|---|---|
| h&#8321; | w = (0,5 ; 0,4) | b = &minus;0,2 |
| h&#8322; | w = (&minus;0,6 ; 0,3) | b = 0,5 |
| y | w = (1,2 ; &minus;0,8) | b = 0,1 |

**Tabel bantu sigmoid:** &sigma;(1,02) &asymp; 0,73 &nbsp;&middot;&nbsp; &sigma;(&minus;0,78) &asymp; 0,31

**Ingat urutannya:** hitung z tiap neuron tersembunyi, terapkan ReLU, baru hitung neuron keluaran.

## Kasus A &mdash; masukan x = (1 ; 2)

In [ ]:
# [PERIKSA] Kasus A — x = (1 ; 2)
zA_h1 = None      # skor mentah h1 sebelum aktivasi
aA_h1 = None      # setelah ReLU
zA_h2 = None
aA_h2 = None
zA_y  = None      # skor mentah neuron keluaran
pA    = None      # setelah sigmoid

print('Soal 1 — Kasus A')
periksa('z pada h1', zA_h1, '30557accbbaa6b5b')
periksa('a pada h1 setelah ReLU', aA_h1, '30557accbbaa6b5b')
periksa('z pada h2', zA_h2, '9b1356fe79c71f53')
periksa('a pada h2 setelah ReLU', aA_h2, '9b1356fe79c71f53')
periksa('z pada neuron keluaran', zA_y, '5d8105ac26d79325')
periksa('p setelah sigmoid', pA, '99d03fc3771fe684', desimal=2)

## Kasus B &mdash; masukan x = (&minus;1 ; 0)

In [ ]:
# [PERIKSA] Kasus B — x = (-1 ; 0)
zB_h1 = None
aB_h1 = None
zB_h2 = None
aB_h2 = None
zB_y  = None
pB    = None

print('Soal 1 — Kasus B')
periksa('z pada h1', zB_h1, '659d17c0fc169cf1')
periksa('a pada h1 setelah ReLU', aB_h1, 'c1299919d8fa4a81')
periksa('z pada h2', zB_h2, '30557accbbaa6b5b')
periksa('a pada h2 setelah ReLU', aB_h2, '30557accbbaa6b5b')
periksa('z pada neuron keluaran', zB_y, '123827d6228bb203')
periksa('p setelah sigmoid', pB, 'a3b9afe536de2b8b', desimal=2)

In [ ]:
# [KODE] Memeriksa perhitungan Anda
def relu(z):
    return np.maximum(0, z)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

W1 = np.array([[0.5, 0.4], [-0.6, 0.3]])     # baris = neuron tersembunyi
b1 = np.array([-0.2, 0.5])
W2 = np.array([1.2, -0.8])
b2 = 0.1

def maju(x, tampilkan=True):
    z1 = W1 @ x + b1
    a1 = relu(z1)
    z2 = W2 @ a1 + b2
    p  = sigmoid(z2)
    if tampilkan:
        print(f'  x         = {x}')
        print(f'  z lapisan tersembunyi = {np.round(z1, 4)}')
        print(f'  a setelah ReLU        = {np.round(a1, 4)}')
        print(f'  z keluaran            = {round(z2, 4)}')
        print(f'  p setelah sigmoid     = {round(p, 4)}')
    return p

print('Kasus A'); maju(np.array([1.0, 2.0])); print()
print('Kasus B'); maju(np.array([-1.0, 0.0]))

**Pertanyaan penalaran 1.1** Pada Kasus B, neuron h&#8321; menghasilkan nilai nol setelah ReLU. Apa akibatnya bagi neuron keluaran? Apakah bobot 1,2 yang menghubungkan h&#8321; ke keluaran ikut berpengaruh pada hasil?

> _Tulis jawaban Anda di sini._

**Pertanyaan penalaran 1.2** Bayangkan h&#8321; menghasilkan nol untuk **hampir seluruh** data latih. Apa yang terjadi pada gradien yang mengalir ke h&#8321; saat backpropagation, dan apa akibatnya bagi bobot neuron itu?

> _Tulis jawaban Anda di sini._

**Pertanyaan penalaran 1.3** Gejala pada pertanyaan sebelumnya punya nama khusus. Sebutkan namanya, dan tuliskan **dua** cara mengatasinya.

> _Tulis jawaban Anda di sini._


---

# Soal 2 &mdash; Mengapa Aktivasi Diperlukan

Sel berikut membandingkan jaringan **dengan** dan **tanpa** fungsi aktivasi.

In [ ]:
# [KODE] Jaringan tanpa aktivasi runtuh menjadi satu operasi linier
A1 = np.array([[0.5, 0.4], [-0.6, 0.3]])
A2 = np.array([[1.2, -0.8], [0.3, 0.9]])
A3 = np.array([[0.7, -0.4]])

x_uji = np.array([2.0, 3.0])

# Tiga lapisan tanpa aktivasi
hasil_berlapis = A3 @ (A2 @ (A1 @ x_uji))

# Ketiga matriks digabung menjadi satu
A_gabung = A3 @ A2 @ A1
hasil_satu_lapis = A_gabung @ x_uji

print('Tiga lapisan tanpa aktivasi :', np.round(hasil_berlapis, 6))
print('Satu matriks gabungan       :', np.round(hasil_satu_lapis, 6))
print()
print('Matriks gabungan:'); print(np.round(A_gabung, 4))

**Pertanyaan penalaran 2.1** Kedua hasil di atas identik. Jelaskan apa yang hal itu buktikan tentang jaringan berlapis **tanpa** fungsi aktivasi.

> _Tulis jawaban Anda di sini._

**Pertanyaan penalaran 2.2** Berdasarkan bukti tersebut, jelaskan mengapa fungsi aktivasi disebut sebagai **satu-satunya sumber kelengkungan** pada neural network.

> _Tulis jawaban Anda di sini._

**Pertanyaan penalaran 2.3** Selain harus non-linier, fungsi aktivasi juga harus **dapat diturunkan**. Mengapa syarat kedua itu diperlukan? Kaitkan dengan Gradient Descent dari pertemuan 4.

> _Tulis jawaban Anda di sini._


---

# Soal 3 &mdash; Membandingkan Fungsi Aktivasi

In [ ]:
# [KODE] Tiga fungsi aktivasi dan turunannya
z = np.linspace(-6, 6, 400)
fungsi = {
    'Sigmoid': (sigmoid(z), sigmoid(z) * (1 - sigmoid(z))),
    'Tanh'   : (np.tanh(z), 1 - np.tanh(z) ** 2),
    'ReLU'   : (relu(z), (z > 0).astype(float)),
}

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for nama, (nilai, turunan) in fungsi.items():
    axes[0].plot(z, nilai, lw=2, label=nama)
    axes[1].plot(z, turunan, lw=2, label=nama)
axes[0].set_title('Nilai fungsi'); axes[1].set_title('Turunan fungsi')
for ax in axes:
    ax.set_xlabel('z'); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

print('Turunan sigmoid pada beberapa titik:')
for titik in [-6, -3, 0, 3, 6]:
    s = sigmoid(titik)
    print(f'  z = {titik:>3} : turunan = {s*(1-s):.6f}')

**Pertanyaan penalaran 3.1** Perhatikan turunan sigmoid pada z = &minus;6 dan z = 6. Berapa nilainya, dan apa akibatnya bagi gradien yang mengalir mundur melalui neuron itu?

> _Tulis jawaban Anda di sini._

**Pertanyaan penalaran 3.2** Gejala tersebut punya nama. Sebutkan namanya, dan jelaskan mengapa ia menjadi lebih parah pada jaringan yang **lebih dalam**.

> _Tulis jawaban Anda di sini._

**Pertanyaan penalaran 3.3** Bandingkan turunan ReLU dengan turunan sigmoid pada sisi positif. Jelaskan mengapa hal itu membuat ReLU menjadi pilihan bawaan untuk lapisan tersembunyi saat ini.

> _Tulis jawaban Anda di sini._


---

# Soal 4 &mdash; Menelusuri Alur Backpropagation

Bagian ini **tidak memerlukan perhitungan**. Yang diuji adalah pemahaman alurnya.

**Pertanyaan penalaran 4.1** Urutkan keempat langkah berikut sesuai urutan yang benar, lalu jelaskan singkat apa yang dikerjakan tiap langkah.

| Langkah (acak) |
|---|
| Menggeser tiap bobot berlawanan arah gradiennya |
| Membagikan galat mundur ke tiap neuron sebanding sumbangannya |
| Menghitung galat pada neuron keluaran |
| Menghitung gradien loss terhadap tiap bobot |

| Urutan yang benar | Yang dikerjakan |
|---|---|
| 1 | |
| 2 | |
| 3 | |
| 4 | |

**Pertanyaan penalaran 4.2** Seorang rekan menyebut backpropagation sebagai “algoritma pelatihan neural network”. Apakah pernyataan itu tepat? Bila tidak, apa peran backpropagation sebenarnya, dan apa yang benar-benar melatih jaringannya?

> _Tulis jawaban Anda di sini._

**Pertanyaan penalaran 4.3** Isi tabel padanan berikut, menghubungkan istilah neural network dengan istilah yang sudah Anda kenal sejak pertemuan 4.

| Istilah pada neural network | Padanannya pada pertemuan 4 |
|---|---|
| Epoch | |
| Mini-batch | |
| Learning rate | |
| Fungsi loss | |


---

# Soal 5 &mdash; Dropout dan Early Stopping

In [ ]:
# [KODE] Kurva pelatihan sebuah jaringan
epoch     = np.arange(0, 100, 10)
loss_latih   = np.array([1.10, 0.72, 0.51, 0.38, 0.29, 0.22, 0.17, 0.13, 0.10, 0.07])
loss_validasi = np.array([1.12, 0.78, 0.60, 0.50, 0.46, 0.47, 0.52, 0.60, 0.70, 0.82])

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(epoch, loss_latih, marker='o', lw=2, label='loss data latih')
ax.plot(epoch, loss_validasi, marker='s', lw=2, label='loss data validasi')
titik_min = epoch[loss_validasi.argmin()]
ax.axvline(titik_min, ls='--', lw=1, color='gray')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Kurva pelatihan')
plt.tight_layout(); plt.show()

In [ ]:
# [PERIKSA] Membaca kurva pelatihan
epoch_terbaik      = None    # epoch dengan loss validasi terendah
loss_validasi_min  = None    # nilai loss validasi terendah

print('Soal 5 — Membaca kurva pelatihan')
periksa('Epoch dengan loss validasi terendah', epoch_terbaik, 'efc024a550f42550')
periksa('Nilai loss validasi terendah', loss_validasi_min, 'ad9dadb817c4ea95', desimal=2)

**Pertanyaan penalaran 5.1** Bila pelatihan dihentikan pada epoch 40, bobot yang disimpan adalah bobot pada epoch berapa? Jelaskan mengapa bukan bobot pada epoch terakhir.

> _Tulis jawaban Anda di sini._

**Pertanyaan penalaran 5.2** Early stopping menuntut adanya **data validasi**. Jelaskan mengapa teknik ini tidak dapat dipakai bila data hanya dibagi menjadi latih dan uji saja.

> _Tulis jawaban Anda di sini._

**Pertanyaan penalaran 5.3** Sebuah model dilatih dengan dropout 0,3. Saat model dipakai memprediksi, hasilnya berbeda-beda untuk masukan yang sama persis. Apa yang salah, dan bagaimana memperbaikinya?

> _Tulis jawaban Anda di sini._


---

## Lembar Catatan untuk Dibawa ke Kelas

### Bagian yang membuat saya tersendat

| Nomor soal | Langkah keberapa saya tersendat | Apa yang membingungkan |
|---|---|---|
| | | |
| | | |

### Pertanyaan yang ingin saya ajukan

1. Berapa banyak lapisan tersembunyi yang cukup untuk sebuah masalah, dan atas dasar apa menentukannya?
2. Mengapa neural network sering kalah dari model berbasis pohon pada data berbentuk tabel?
3. Apa yang terjadi bila learning rate terlalu besar pada jaringan yang dalam?

> _Tambahkan pertanyaan Anda sendiri di sini._

---

## Bagaimana Latihan Ini Dinilai

Berkas ini **tidak dinilai**. Yang dinilai adalah kontribusi Anda saat pembahasan di kelas, memakai rubrik Aktivitas Partisipatif (Lampiran 3a RPS).

| Tingkat capaian | Deskriptor |
|---|---|
| Sangat Baik (85–100) | Berkontribusi konsisten dengan argumen berdasar, mengaitkan materi dengan kasus, dan mendorong diskusi maju |
| Baik (70–84) | Berkontribusi teratur dengan argumen relevan dan tepat |
| Cukup (55–69) | Berkontribusi sesekali, argumen relevan namun dangkal |
| Kurang (kurang dari 55) | Jarang atau tidak berkontribusi |

**Menyampaikan di mana Anda tersendat termasuk kontribusi.** Mengaku belum paham pada bagian tertentu dinilai lebih tinggi daripada diam.
